# Prompt Registry

https://mlflow.org/docs/latest/genai/prompt-registry/

# 1.) Set-up

In [1]:
import mlflow

mlflow.set_tracking_uri('http://localhost:5000')


k:\Gen AI\project\restaurant-rag\restaurant_rag_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


---------------------------------------------

# 2.) Prompt Version 1

create a new prompt using the Python API, use mlflow.genai.register_prompt() API

In [2]:
V1_ASSESS_SYSTEM_PROMPT = """\
You are a restaurant operations analyst. You get one customer review plus similar
past reviews from this restaurant's corpus.

Ground every claim. Sentiment and issues come from the new review. Any claim that
an issue is recurring must cite a retrieved review; if none support it, say the
corpus doesn't show it. Never invent facts about staffing, suppliers, or prices.
If retrieved reviews are unrelated, say so and assess the review alone.

Reply in the review's language. Be brief — an operator reads this."""


# Use double curly braces for variables in the template
V1_ASSESS_USER_TEMPLATE = """\
<past_reviews>
{{ user_review }}
</past_reviews>

<review>
{{ past_review }}
</review>

Assess in exactly these 5 lines:

1. Sentiment: positive/negative/mixed/neutral + confidence 0-1 + the phrase driving it
2. Issues: bullets, each tagged [taste|portion|freshness|service|wait|price|cleanliness|ambience]. "none" if none
3. Precedent: per issue — recurring / one case / not in corpus, + source id
4. Actions: if negative/mixed, 2-3 fixes tied to issue numbers. Tag guesses "verify:". If positive, what to protect
5. Priority: high/med/low + one clause. High = food safety or repeated failure"""


# Register a new prompt
chat_prompt_template = [
    {
        'role': 'system',
        'content': V1_ASSESS_SYSTEM_PROMPT,
    },
    {
        'role': 'user',
        'content': V1_ASSESS_USER_TEMPLATE,
    },
]

prompt_obj = mlflow.genai.register_prompt(
    name='assess-rag-prompt',
    template=chat_prompt_template,
    commit_message='Initial commit with system and user roles', # Optional: Provide a commit message to describe the changes
    tags={
        'author': 'issara.wbk@gmail.com',
        'task': 'assessment',
        'type': 'chat',
        'language': 'en',
    }
)


2026/11/23 15:43:48 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: assess-rag-prompt, version 1


In [3]:
prompt_obj

PromptVersion(name=assess-rag-prompt, version=1, template=[{"role": "system", "content":...)

In [4]:
# The prompt object contains information about the registered prompt
print(f"Created prompt '{prompt_obj.name}' (version {prompt_obj.version})")

Created prompt 'assess-rag-prompt' (version 1)


# 3.) Prompt Version 2

In [5]:
V2_ASSESS_SYSTEM_PROMPT = """\
You are a restaurant operations analyst. You get one customer review plus similar
past reviews from this restaurant's corpus.

Ground every claim. Sentiment and issues come from the new review. Any claim that
an issue is recurring must cite a retrieved review; if none support it, say the
corpus doesn't show it. Never invent facts about staffing, suppliers, or prices.
If retrieved reviews are unrelated, say so and assess the review alone.

Reply in the review's language. Be brief — an operator reads this."""

V2_ASSESS_USER_TEMPLATE = """\
<1st-shot example>
Review: ร้านอยู่ในเมืองแม่ฮ่องสอน บรรยากาศเงียบสงบ วันธรรมดาคนไม่เยอะ มีดนตรีสด แขกส่วนใหญ่ที่เห็นจะเป็นฝรั่ง(วันธรรมดานะ) อากาศไม่ร้อน การบริการดี พนักงานแนะนำอาหารดีมาก มื้อนี้ทานกันเบาๆเพราะมา 2 คน ธรรมดาก็มาสองคนแต่สั่งเหมือนมา 4 คน เหลือทุกที ครั้งนี้เลยเตือนตัวเองสั่งแค่พอทานพอ 2 เมนู ยำผักกูดอาหารแนะนำของร้าน เราชอบมาก แต่แฟนเราไม่ชอบเลย ทางร้านทำออกมาแตกต่างจากร้านอื่นเหมือนใช้กระทิในการทาน มีพริกซอย มีกุนเชียงชิ้นเล็กๆ มีหมูสับ ทานรวมกันแล้วอร่อยมากค่ะ อีกเมนูจำชื่อเมนูไม่ได้เพราะนานแล้ว แต่เป็นเมนูที่ทางร้านแนะนำอีกเหมือนกัน อร่อยเนื้อปลาสดไม่เหม็นคาว ลองมาทานกันเนอะ แม่ฮ่องสอนร้านอาหารไม่ค่อยเยอะ หาไม่ยากค่ะ
Result:
1) Sentiment: positive; confidence 0.75; driving phrases: อร่อยมากค่ะ, บริการดีมาก  
2) Issues: none  
3) Precedent: none; corpus unrelated (ไม่พบรีวิวที่เกี่ยวข้องกับร้านนี้ในชุดข้อมูล)  
4) Actions: รักษาคุณภาพอาหารและความสด; รักษาบริการและการต้อนรับให้ดี; คงบรรยากาศเงียบสงบและดนตรีสดให้ต่อไป  
5) Priority: low (ไม่มีความเสี่ยงด้านความปลอดภัย เน้นรักษาคุณภาพอาหารและบริการ)
</1st-shot example>

<2nd-shot example>
Review: คือต้องบอกก่อนว่าเมนูที่สั่งวันนี้ไม่มีเนื้อแพงๆรวมอยู่ด้วย เลยไม่ขอรีวิวคุณภาพเนื้อของเค้าค่ะ เพราะในโต๊ะไม่กินเนื้อกันเลยสั่งเป็นเมนูธรรมดา เบอเกอร์หมู พาสต้า ซี่โครง ส่วนตัวรสชาติทั่วๆไป ทานได้ ไม่มีอันไหนอร่อยเป็นพิเศษ คิดว่าร้านดังมาก น่าจะเป็นที่คุณภาพเนื้อมากกว่าค่ะ
Result:
1) Sentiment: neutral; confidence 0.70; driving phrase: รสชาติทั่วๆไป ทานได้ ไม่มีอันไหนอร่อยเป็นพิเศษ
2) Issues: [taste] เมนูที่ไม่ใช่เนื้อ (เบอร์เกอร์หมู พาสต้า ซี่โครง) รสชาติธรรมดา ไม่โดดเด่น
3) Precedent: none; corpus unrelated (ไม่พบรีวิวที่เกี่ยวข้องกับร้านนี้ในชุดข้อมูล)  
4) Actions: verify: ทดสอบและปรับปรุงรสชาติเมนูหมู/พาสต้า/ซี่โครงให้มีจุดเด่นมากขึ้น; verify: สื่อสารเมนูซิกเนเจอร์ของร้านให้ชัดเจนสำหรับลูกค้าที่ไม่ทานเนื้อ
5) Priority: low (รสชาติอยู่ในระดับทานได้ ไม่มีปัญหาความปลอดภัยหรือข้อร้องเรียนรุนแรง)
</2nd-shot example>

<3rd-shot example>
Review: เป็นร้านที่ที่บ้านพามากินบ่อยๆ กินมานานมากแล้ว รสชาติอาหาร และคุณภาพ รู้สึกว่าลดลงเรื่อยๆ พนักงานบริการไม่ค่อยดี ฟังไม่ค่อยรู้เรื่อง จดผิดจดถูก ขนมหวาน สั่งบัวลอยเผือกมาทำแบบไม่ตั้งใจ เหมือนเอาแช่แข็งไว้แล้วแค่ไปอุ่นร้อน ตอนเสิร์ฟเหมือนไม่ได้เช็คว่ามันโอเครึยัง พอกินเข้าไปเย็นและแข็งอย่างกับก้อนหิน ปรับปรุงด่วนค่าาาา!!!
Result:
1) Sentiment: negative; confidence 0.75; driving phrase: รสชาติและคุณภาพลดลงเรื่อยๆ และบริการไม่ค่อยดี
2) Issues: [taste] รสชาติ/คุณภาพอาหารลดลง; [service] พนักงานบริการไม่ค่อยดี/จดผิด; [freshness] ของหวาน/อาหารบางจานเย็น/แข็ง เหมือนถูกแช่แข็ง
3) Precedent: [taste] past_review at the indices 3 and 4; [service] past_review at the indices 0 and 2; [freshness] past_review at the index 2
4) Actions: verify: ฝึกอบรมพนักงาน, ยืนยันออเดอร์กับลูกค้าก่อนเสิร์ฟ, ตรวจสอบคำสั่งทุกครั้ง; verify: มาตรฐานสูตรอาหาร ควบคุมส่วนและรสชาติ, ตรวจอุณหภูมิอาหารก่อนเสิร์ฟ; verify: ควบคุมการเก็บรักษา/ละลาย/อุ่นร้อน, บันทึกอุณหภูมิวัสดุอาหาร
5) Priority: high (เนื่องจากปัญหาซ้ำซากด้านบริการ/รสชาติและคุณภาพส่งผลกระทบประสบการณ์ลูกค้า)
</3rd-shot example>

<instructions>
Assess the review, which is enclosed between <review> tags below, into the exact following 5 lines, where the text enclosed by the <past_reviews> tags is past review(s) for context that is retrieved from database:
1. Sentiment: positive/negative/mixed/neutral + confidence 0.0-1.0 + the phrase driving it
2. Issues: bullets, each tagged [taste|portion|freshness|service|wait|price|cleanliness|ambience]. "none" if none
3. Precedent: per issue — recurring / one case / not in corpus, + source id
4. Actions: if negative/mixed, 2-3 fixes tied to issue numbers. Tag guesses "verify:". If positive, what to protect
5. Priority: high/medium/low + one clause. High = food safety or repeated failure
</instructions>

<review>
{{ user_review }}
</review>

<past_reviews>
{{ past_review }}
</past_reviews>
"""


In [ ]:
v2_chat_prompt_template = [
    {
        'role': 'system',
        'content': V2_ASSESS_SYSTEM_PROMPT
    },
    {
        'role': 'user',
        'content': V2_ASSESS_USER_TEMPLATE
    }
]

v2_prompt_obj = mlflow.genai.register_prompt(
    name='assess-rag-prompt',
    template=v2_chat_prompt_template,
    commit_message='version 2 of assessment rag prompt (few-shot prompting and other basic best practices of prompting)',
    tags={
        'author': 'issara.wbk@gmail.com',
        'task': 'assessment',
        'type': 'chat',
        'language': 'en',
    },
)

v2_prompt_obj

2026/11/23 15:43:59 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for prompt version to finish creation. Prompt name: assess-rag-prompt, version 2


PromptVersion(name=assess-rag-prompt, version=2, template=[{"role": "system", "content":...)

In [7]:
print(f"Created prompt '{v2_prompt_obj.name}' (version {v2_prompt_obj.version})")

Created prompt 'assess-rag-prompt' (version 2)


# Prompt Version 3

In [ ]:
V3_ASSESS_SYSTEM_PROMPT = """\
You are a restaurant operations analyst. You get one customer review plus similar
past reviews from this restaurant's corpus.

Ground every claim. Sentiment and issues come from the new review. Any claim that
an issue is recurring must cite a retrieved review; if none support it, say the
corpus doesn't show it. Never invent facts about staffing, suppliers, or prices.
If retrieved reviews are unrelated, say so and assess the review alone.

Reply in the review's language. Be brief — an operator reads this."""

V3_ASSESS_USER_TEMPLATE = """\
<1st-shot example>
Review: ร้านอยู่ในเมืองแม่ฮ่องสอน บรรยากาศเงียบสงบ วันธรรมดาคนไม่เยอะ มีดนตรีสด แขกส่วนใหญ่ที่เห็นจะเป็นฝรั่ง(วันธรรมดานะ) อากาศไม่ร้อน การบริการดี พนักงานแนะนำอาหารดีมาก มื้อนี้ทานกันเบาๆเพราะมา 2 คน ธรรมดาก็มาสองคนแต่สั่งเหมือนมา 4 คน เหลือทุกที ครั้งนี้เลยเตือนตัวเองสั่งแค่พอทานพอ 2 เมนู ยำผักกูดอาหารแนะนำของร้าน เราชอบมาก แต่แฟนเราไม่ชอบเลย ทางร้านทำออกมาแตกต่างจากร้านอื่นเหมือนใช้กระทิในการทาน มีพริกซอย มีกุนเชียงชิ้นเล็กๆ มีหมูสับ ทานรวมกันแล้วอร่อยมากค่ะ อีกเมนูจำชื่อเมนูไม่ได้เพราะนานแล้ว แต่เป็นเมนูที่ทางร้านแนะนำอีกเหมือนกัน อร่อยเนื้อปลาสดไม่เหม็นคาว ลองมาทานกันเนอะ แม่ฮ่องสอนร้านอาหารไม่ค่อยเยอะ หาไม่ยากค่ะ
Result:
1) Sentiment: positive; confidence 0.75; driving phrases: อร่อยมากค่ะ, บริการดีมาก  
2) Issues: none  
3) Precedent: none; corpus unrelated (ไม่พบรีวิวที่เกี่ยวข้องกับร้านนี้ในชุดข้อมูล)  
4) Actions: รักษาคุณภาพอาหารและความสด; รักษาบริการและการต้อนรับให้ดี; คงบรรยากาศเงียบสงบและดนตรีสดให้ต่อไป  
5) Priority: low (ไม่มีความเสี่ยงด้านความปลอดภัย เน้นรักษาคุณภาพอาหารและบริการ)
</1st-shot example>

<2nd-shot example>
Review: คือต้องบอกก่อนว่าเมนูที่สั่งวันนี้ไม่มีเนื้อแพงๆรวมอยู่ด้วย เลยไม่ขอรีวิวคุณภาพเนื้อของเค้าค่ะ เพราะในโต๊ะไม่กินเนื้อกันเลยสั่งเป็นเมนูธรรมดา เบอเกอร์หมู พาสต้า ซี่โครง ส่วนตัวรสชาติทั่วๆไป ทานได้ ไม่มีอันไหนอร่อยเป็นพิเศษ คิดว่าร้านดังมาก น่าจะเป็นที่คุณภาพเนื้อมากกว่าค่ะ
Result:
1) Sentiment: neutral; confidence 0.70; driving phrase: รสชาติทั่วๆไป ทานได้ ไม่มีอันไหนอร่อยเป็นพิเศษ
2) Issues: [taste] เมนูที่ไม่ใช่เนื้อ (เบอร์เกอร์หมู พาสต้า ซี่โครง) รสชาติธรรมดา ไม่โดดเด่น
3) Precedent: none; corpus unrelated (ไม่พบรีวิวที่เกี่ยวข้องกับร้านนี้ในชุดข้อมูล)  
4) Actions: verify: ทดสอบและปรับปรุงรสชาติเมนูหมู/พาสต้า/ซี่โครงให้มีจุดเด่นมากขึ้น; verify: สื่อสารเมนูซิกเนเจอร์ของร้านให้ชัดเจนสำหรับลูกค้าที่ไม่ทานเนื้อ
5) Priority: low (รสชาติอยู่ในระดับทานได้ ไม่มีปัญหาความปลอดภัยหรือข้อร้องเรียนรุนแรง)
</2nd-shot example>

<3rd-shot example>
Review: เป็นร้านที่ที่บ้านพามากินบ่อยๆ กินมานานมากแล้ว รสชาติอาหาร และคุณภาพ รู้สึกว่าลดลงเรื่อยๆ พนักงานบริการไม่ค่อยดี ฟังไม่ค่อยรู้เรื่อง จดผิดจดถูก ขนมหวาน สั่งบัวลอยเผือกมาทำแบบไม่ตั้งใจ เหมือนเอาแช่แข็งไว้แล้วแค่ไปอุ่นร้อน ตอนเสิร์ฟเหมือนไม่ได้เช็คว่ามันโอเครึยัง พอกินเข้าไปเย็นและแข็งอย่างกับก้อนหิน ปรับปรุงด่วนค่าาาา!!!
Result:
1) Sentiment: negative; confidence 0.75; driving phrase: รสชาติและคุณภาพลดลงเรื่อยๆ และบริการไม่ค่อยดี
2) Issues: [taste] รสชาติ/คุณภาพอาหารลดลง; [service] พนักงานบริการไม่ค่อยดี/จดผิด; [freshness] ของหวาน/อาหารบางจานเย็น/แข็ง เหมือนถูกแช่แข็ง
3) Precedent: [taste] past_review at the indices 3 and 4; [service] past_review at the indices 0 and 2; [freshness] past_review at the index 2
4) Actions: verify: ฝึกอบรมพนักงาน, ยืนยันออเดอร์กับลูกค้าก่อนเสิร์ฟ, ตรวจสอบคำสั่งทุกครั้ง; verify: มาตรฐานสูตรอาหาร ควบคุมส่วนและรสชาติ, ตรวจอุณหภูมิอาหารก่อนเสิร์ฟ; verify: ควบคุมการเก็บรักษา/ละลาย/อุ่นร้อน, บันทึกอุณหภูมิวัสดุอาหาร
5) Priority: high (เนื่องจากปัญหาซ้ำซากด้านบริการ/รสชาติและคุณภาพส่งผลกระทบประสบการณ์ลูกค้า)
</3rd-shot example>

<instructions>
Assess the review, which is enclosed between <review> tags below, into the exact following 5 lines, where the text enclosed by the <past_reviews> tags is past review(s) for context that is retrieved from database:
1. Sentiment: positive/negative/mixed/neutral + confidence 0.0-1.0 + the phrase driving it
2. Issues: bullets, each tagged [taste|portion|freshness|service|wait|price|cleanliness|ambience]. "none" if none
3. Precedent: per issue — recurring / one case / not in corpus, + source id
4. Actions: if negative/mixed, 2-3 fixes tied to issue numbers. Tag guesses "verify:". If positive, what to protect
5. Priority: high/medium/low + one clause. High = food safety or repeated failure
</instructions>

<review>
{{ user_review }}
</review>

<past_reviews>
{{ past_review }}
</past_reviews>
"""


# 4.) Load and Use the Prompt

To use a prompt in your LLM application or AI agent, you can load it with the mlflow.genai.load_prompt() API and fill in the variables using the mlflow.entities.model_registry.PromptVersion.format() method of the prompt object.

In [8]:
v1_loaded_prompt = mlflow.genai.load_prompt('prompts:/assess-rag-prompt/1')
v2_loaded_prompt = mlflow.genai.load_prompt('prompts:/assess-rag-prompt/2')

v1_loaded_prompt

PromptVersion(name=assess-rag-prompt, version=1, template=[{"role": "system", "content":...)

In [9]:
v2_loaded_prompt

PromptVersion(name=assess-rag-prompt, version=2, template=[{"role": "system", "content":...)

In [11]:
v1_loaded_prompt.format(user_review='test past review', past_review='test user review')

[{'role': 'system',
  'content': "You are a restaurant operations analyst. You get one customer review plus similar\npast reviews from this restaurant's corpus.\n\nGround every claim. Sentiment and issues come from the new review. Any claim that\nan issue is recurring must cite a retrieved review; if none support it, say the\ncorpus doesn't show it. Never invent facts about staffing, suppliers, or prices.\nIf retrieved reviews are unrelated, say so and assess the review alone.\n\nReply in the review's language. Be brief — an operator reads this."},
 {'role': 'user',
  'content': '<past_reviews>\ntest past review\n</past_reviews>\n\n<review>\ntest user review\n</review>\n\nAssess in exactly these 5 lines:\n\n1. Sentiment: positive/negative/mixed/neutral + confidence 0-1 + the phrase driving it\n2. Issues: bullets, each tagged [taste|portion|freshness|service|wait|price|cleanliness|ambience]. "none" if none\n3. Precedent: per issue — recurring / one case / not in corpus, + source id\n4. 

Use the prompt with an LLM

In [21]:
import openai
from dotenv import load_dotenv

load_dotenv("../env/.env")  

client = openai.OpenAI()
response = client.chat.completions.create(
    messages=v2_loaded_prompt.format(user_review='แค่กัดไก่ก็รู้สึกอร่อยทันที ไม่รู้เพราะอะไร', 
                                     past_review='ไก่อร่อยเพราะเนื้อฉ่ำ'),
    model="gpt-5-nano",
)

print(response.choices[0].message.content)

Sentiment: positive; confidence 0.75; driving phrase: แค่กัดไก่ก็รู้สึกอร่อยทันที
Issues: none
Precedent: one case; source: past_reviews[0]
Actions: verify: รักษารสชาติและความฉ่ำของไก่ให้คงที่; verify: สื่อสารจุดเด่นของไก่ในเมนูให้ลูกค้าตระหนัก
Priority: low; เน้นรักษาคุณภาพรสชาติไก่ให้สม่ำเสมอ


In [22]:
print(response.model_dump_json(indent=2))

{
  "id": "chatcmpl-EnKGunRZq34IMGbhboXtczy60Lo4n",
  "choices": [
    {
      "finish_reason": "stop",
      "index": 0,
      "logprobs": null,
      "message": {
        "content": "Sentiment: positive; confidence 0.75; driving phrase: แค่กัดไก่ก็รู้สึกอร่อยทันที\nIssues: none\nPrecedent: one case; source: past_reviews[0]\nActions: verify: รักษารสชาติและความฉ่ำของไก่ให้คงที่; verify: สื่อสารจุดเด่นของไก่ในเมนูให้ลูกค้าตระหนัก\nPriority: low; เน้นรักษาคุณภาพรสชาติไก่ให้สม่ำเสมอ",
        "refusal": null,
        "role": "assistant",
        "annotations": [],
        "audio": null,
        "function_call": null,
        "tool_calls": null
      }
    }
  ],
  "created": 1795424704,
  "model": "gpt-5-nano-2025-08-07",
  "object": "chat.completion",
  "moderation": null,
  "service_tier": "default",
  "system_fingerprint": null,
  "usage": {
    "completion_tokens": 2175,
    "prompt_tokens": 1551,
    "total_tokens": 3726,
    "completion_tokens_details": {
      "accepted_prediction_

In [23]:
1

1

In [ ]:
# 1. Define a test dataset
eval_data = [
    {"inputs": {"question": "What is MLflow Prompt Registry?"}},
    {"inputs": {"question": "How do aliases work in MLflow?"}},
]

# 2. Wrap a specific prompt version in your prediction function
def predict_v1(question: str) -> str:
    prompt = mlflow.genai.load_prompt("prompts:/qa_prompt/1")  # Pin to v1
    messages = prompt.format(question=question)
    response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
    return response.choices[0].message.content

# 3. Run evaluation — MLflow logs all responses and links them to v1
mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_v1,
    scorers=[],  # Even with no scorers, inputs, responses, and traces are logged
)

# 5.) Evaluation

Evaluates `prompts:/assess-rag-prompt/1` and `prompts:/assess-rag-prompt/2` on
[`data/eval/reviews_heldout_sample.jsonl`](../data/eval/reviews_heldout_sample.jsonl) using:

1. **Project retriever** (`app.services.milvus_service.search`, cached per review so embeddings are paid once across v1 and v2).
2. **`mlflow.genai.evaluate`** with a `RETRIEVER`-traced pipeline, active-model prompt linking, deterministic format checks (`evaluation/extractive.py`), and an OpenAI-SDK groundedness judge (`gpt-4o-mini`, no `litellm` needed).
3. **Project RAGAS scoring** (`Faithfulness` on full 5 lines & extractive lines 1–3 with `[review] + contexts`, plus `LLMContextPrecisionWithoutReference`), logged into each version's MLflow run for direct comparison.

> **Note on v1 vs v2:** In `assess-rag-prompt` v1, `<past_reviews>` wraps `{{ user_review }}` and `<review>` wraps `{{ past_review }}` (swapped). Version 2 fixes that swap and adds 3-shot Thai examples. Keep `N_MAX = 10` first to verify cost and Milvus connectivity before scaling to `100`.

In [1]:
import json
import re
import sys
from pathlib import Path

import mlflow
import pandas as pd
from dotenv import load_dotenv
from mlflow.entities import Document, Feedback, SpanType
from mlflow.genai.scorers import scorer
from openai import OpenAI

# Prevent cp1252 UnicodeEncodeError when MLflow prints run emojis on Windows.
if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Resolve repo root whether notebook CWD is `notebook/` or the repo root.
REPO_ROOT = Path.cwd().parent if (Path.cwd().parent / "app").is_dir() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

load_dotenv(REPO_ROOT / "env" / ".env")

from app.config import settings
from app.services.milvus_service import search as milvus_search
from app.services.rag_service import NO_CONTEXT_ANSWER
from evaluation.extractive import extractive_only, sentiment_label

# ---- Evaluation config ----
PROMPT_NAME = "assess-rag-prompt"
PROMPT_VERSIONS = [1, 2]
EVAL_FILE = REPO_ROOT / "data" / "eval" / "reviews_heldout_sample.jsonl"
N_MAX = 100                    # Raise to 100 for the full held-out sample
TOP_K = settings.top_k        # 5
GENERATOR_MODEL = settings.llm_model
JUDGE_MODEL = "gpt-5.4-nano"   # Used by direct OpenAI judge & RAGAS (gpt-5-* uses default temperature=1)

mlflow.set_tracking_uri("http://localhost:5000")
mlflow.set_experiment("restaurant_rag/prompt_registry_eval")

eval_rows = sorted(
    (json.loads(line) for line in EVAL_FILE.read_text(encoding="utf-8").splitlines() if line.strip()),
    key=lambda r: r["id"],
)[:N_MAX]

eval_data = [{"inputs": {"review": row["text"]}} for row in eval_rows]

client = OpenAI(api_key=settings.openai_api_key or None)
print(f"Repo root : {REPO_ROOT}")
print(f"Eval rows : {len(eval_rows)} from {EVAL_FILE.name}")
print(f"Milvus    : {settings.milvus_uri} ({settings.milvus_database}/{settings.milvus_collection})")
print(f"Generator : {GENERATOR_MODEL} | Judge: {JUDGE_MODEL}")

k:\Gen AI\project\restaurant-rag\restaurant_rag_venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
DASHSCOPE_API_KEY not set. Qwen service will fail on generate_answer().
2026/11/24 14:29:13 INFO mlflow.tracking.fluent: Experiment with name 'restaurant_rag/prompt_registry_eval' does not exist. Creating a new experiment.


Repo root : k:\Gen AI\project\restaurant-rag
Eval rows : 100 from reviews_heldout_sample.jsonl
Milvus    : http://localhost:19530 (restaurant_db/wongnai_review_coll)
Generator : gpt-5-nano | Judge: gpt-5.4-nano


In [2]:
# Cache Milvus retrieval per review so v1 and v2 see identical contexts and we
# only pay for each query embedding once.
_retrieval_cache: dict[str, list[dict]] = {}
_rag_records_by_version: dict[int, list[dict]] = {}


@mlflow.trace(span_type=SpanType.RETRIEVER)
def retrieve_past_reviews(review: str, top_k: int = TOP_K) -> list[Document]:
    """Retrieve similar past reviews from Milvus as MLflow Document objects."""
    if review not in _retrieval_cache:
        _retrieval_cache[review] = milvus_search(review, top_k=top_k)
    hits = _retrieval_cache[review]
    return [
        Document(
            page_content=hit["text"],
            metadata={
                "source": hit.get("source", ""),
                "doc_uri": hit.get("source", ""),
                "score": hit.get("score"),
            },
        )
        for hit in hits
    ]


def make_predict_fn(version: int):
    """Build a traced RAG predict_fn pinned to a specific MLflow prompt version."""
    _rag_records_by_version[version] = []

    @mlflow.trace
    def predict(review: str) -> str:
        docs = retrieve_past_reviews(review, top_k=TOP_K)
        contexts = [doc.page_content for doc in docs]

        if not contexts:
            answer = NO_CONTEXT_ANSWER
        else:
            prompt = mlflow.genai.load_prompt(f"prompts:/{PROMPT_NAME}/{version}")
            context_block = "\n\n".join(contexts)
            messages = prompt.format(user_review=review, past_review=context_block)
            resp = client.chat.completions.create(
                model=GENERATOR_MODEL,
                messages=messages,
            )
            answer = (resp.choices[0].message.content or "").strip()

        _rag_records_by_version[version].append({
            "user_input": review,
            "response": answer,
            "retrieved_contexts": contexts,
        })
        return answer

    return predict


# ---- MLflow GenAI Scorers (deterministic + OpenAI-SDK judge, no litellm needed) ----
_LINE_NUM_RE = re.compile(r"^\s*([1-5])[.)]\s+", re.M)


@scorer
def extractive_lines_count(outputs: str) -> int:
    """Count Sentiment/Issues/Precedent lines kept by evaluation.extractive.extractive_only (0-3)."""
    sliced = extractive_only(outputs or "")
    return len([ln for ln in sliced.splitlines() if ln.strip()])


@scorer
def has_five_lines(outputs: str) -> bool:
    """Check whether all 5 numbered assessment lines (1..5) are present."""
    found = set(_LINE_NUM_RE.findall(outputs or ""))
    return found == {"1", "2", "3", "4", "5"}


@scorer
def sentiment_parsed(outputs: str) -> bool:
    """True when line 1 yields a valid sentiment label via evaluation.extractive.sentiment_label."""
    return sentiment_label(outputs or "") != "unparsed"


@scorer
def llm_groundedness_and_format(inputs: dict, outputs: str, trace) -> Feedback:
    """Judge whether Sentiment/Issues/Precedent are grounded in the review + retrieved past reviews."""
    review = inputs.get("review", "")
    contexts = _retrieval_cache.get(review, [])
    past_block = "\n---\n".join(f"[{i}] {c['text']}" for i, c in enumerate(contexts))

    judge_prompt = f"""You are grading a restaurant review analyst report.
Check whether lines 1-3 (Sentiment, Issues, Precedent) are strictly supported by the
customer review and the retrieved past reviews, without inventing facts or false precedents.

Customer review:
{review}

Retrieved past reviews:
{past_block}

Analyst report:
{outputs}

Reply ONLY with valid JSON: {{"grounded": true/false, "score": 0.0-1.0, "reason": "one short sentence"}}"""

    resp = client.chat.completions.create(
        model=JUDGE_MODEL,
        response_format={"type": "json_object"},
        messages=[{"role": "user", "content": judge_prompt}],
    )
    verdict = json.loads(resp.choices[0].message.content)
    return Feedback(
        value=float(verdict["score"]),
        rationale=verdict.get("reason", ""),
        metadata={"grounded": str(verdict.get("grounded"))},
    )


SCORERS = [
    extractive_lines_count,
    has_five_lines,
    sentiment_parsed,
    llm_groundedness_and_format,
]

In [3]:
# Run mlflow.genai.evaluate for both prompt versions (links traces & runs to each version)
eval_results = {}

for version in PROMPT_VERSIONS:
    active_model = mlflow.set_active_model(name=f"{PROMPT_NAME}-v{version}")
    print(f"\n=== Evaluating {PROMPT_NAME} v{version} (N={len(eval_data)}) ===")
    res = mlflow.genai.evaluate(
        data=eval_data,
        predict_fn=make_predict_fn(version),
        scorers=SCORERS,
        model_id=active_model.model_id,
    )
    eval_results[version] = res
    print(f"Run ID: {res.run_id}")
    print("Metrics:", res.metrics)

2026/11/24 14:29:50 INFO mlflow.tracking.fluent: LoggedModel with name 'assess-rag-prompt-v1' does not exist, creating one...


2026/11/24 14:29:51 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-4d5ff7d4fc2d4804b7dff30419885e8e



=== Evaluating assess-rag-prompt v1 (N=100) ===


2026/11/24 14:29:52 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-4d5ff7d4fc2d4804b7dff30419885e8e
2026/11/24 14:29:52 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/11/24 14:29:54 INFO mlflow.models.evaluation.utils.trace: Auto tracing is temporarily enabled during the model evaluation for computing some metrics and debugging. To disable tracing, call `mlflow.autolog(disable=True)`.
2026/11/24 14:29:54 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 100/100 [Elapsed: 02:35, Remaining: 00:00] [predict_fn: 83%, scorers: 17%]


2026/11/24 14:33:25 INFO mlflow.tracking.fluent: LoggedModel with name 'assess-rag-prompt-v2' does not exist, creating one...


Run ID: 8ceb05987ecc49be83fef1bc8ac6895f
Metrics: {'sentiment_parsed/mean': np.float64(0.67), 'extractive_lines_count/mean': np.float64(2.9), 'llm_groundedness_and_format/mean': np.float64(0.44480000000000003), 'has_five_lines/mean': np.float64(0.57)}


2026/11/24 14:33:25 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-59e1a85e55554af792b25374eac7a021



=== Evaluating assess-rag-prompt v2 (N=100) ===


2026/11/24 14:33:25 INFO mlflow.tracking.fluent: Active model is set to the logged model with ID: m-59e1a85e55554af792b25374eac7a021
2026/11/24 14:33:25 INFO mlflow.tracking.fluent: Use `mlflow.set_active_model` to set the active model to a different one if needed.
2026/11/24 14:33:26 INFO mlflow.genai.utils.data_validation: Testing model prediction with the first sample in the dataset. To disable this check, set the MLFLOW_GENAI_EVAL_SKIP_TRACE_VALIDATION environment variable to True.
Evaluating: 100%|██████████| 100/100 [Elapsed: 02:13, Remaining: 00:00] [predict_fn: 86%, scorers: 14%]


Run ID: 4652ef7540c444cf923e90c7ccbe0b29
Metrics: {'sentiment_parsed/mean': np.float64(1.0), 'has_five_lines/mean': np.float64(0.17), 'extractive_lines_count/mean': np.float64(3.0), 'llm_groundedness_and_format/mean': np.float64(0.46140000000000003)}


In [4]:
# Run the project's RAGAS metrics (Faithfulness full + extractive, Context Precision)
# and log them back onto each version's MLflow evaluation run.
from langchain_openai import ChatOpenAI
from ragas import EvaluationDataset, evaluate as ragas_evaluate
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import Faithfulness, LLMContextPrecisionWithoutReference


class PinnedTempChatOpenAI(ChatOpenAI):
    """Drops `temperature` so gpt-5-* judges also work if JUDGE_MODEL is switched."""
    def _get_request_payload(self, *args, **kwargs):
        payload = super()._get_request_payload(*args, **kwargs)
        payload.pop("temperature", None)
        return payload


ragas_judge = LangchainLLMWrapper(
    PinnedTempChatOpenAI(model=JUDGE_MODEL, api_key=settings.openai_api_key or None)
)

summary_rows = []

for version in PROMPT_VERSIONS:
    recs = _rag_records_by_version[version]
    full_ds = EvaluationDataset.from_list(recs)
    full_res = ragas_evaluate(
        dataset=full_ds,
        metrics=[LLMContextPrecisionWithoutReference(), Faithfulness()],
        llm=ragas_judge,
    )
    full_df = full_res.to_pandas()
    prec_col = [c for c in full_df.columns if "precision" in c or "utilization" in c][0]

    # Extractive slice: lines 1-3 (Sentiment/Issues/Precedent) grounded on [review] + retrieved chunks
    extractive_recs = [
        {
            "user_input": r["user_input"],
            "response": s,
            "retrieved_contexts": [r["user_input"]] + r["retrieved_contexts"],
        }
        for r in recs
        if (s := extractive_only(r["response"])).strip()
    ]
    if extractive_recs:
        ext_res = ragas_evaluate(
            dataset=EvaluationDataset.from_list(extractive_recs),
            metrics=[Faithfulness()],
            llm=ragas_judge,
        )
        faith_extractive = float(ext_res.to_pandas()["faithfulness"].mean())
    else:
        faith_extractive = float("nan")

    ragas_metrics = {
        "context_precision": float(full_df[prec_col].mean()),
        "faithfulness_full": float(full_df["faithfulness"].mean()),
        "faithfulness_extractive": faith_extractive,
        "n_extractive": len(extractive_recs),
    }

    # Log RAGAS scores & params onto the existing MLflow run created by mlflow.genai.evaluate
    with mlflow.start_run(run_id=eval_results[version].run_id):
        mlflow.log_params({
            "prompt_name": PROMPT_NAME,
            "prompt_version": version,
            "generator_model": GENERATOR_MODEL,
            "judge_model": JUDGE_MODEL,
            "top_k": TOP_K,
            "n_eval": len(recs),
            "n_extractive": len(extractive_recs),
        })
        mlflow.log_metrics({
            k: v for k, v in ragas_metrics.items()
            if k != "n_extractive" and pd.notna(v)
        })

    summary_rows.append({
        "prompt_version": f"v{version}",
        **{k: round(v, 4) if isinstance(v, float) and pd.notna(v) else v for k, v in eval_results[version].metrics.items()},
        **{k: round(v, 4) if isinstance(v, float) and pd.notna(v) else v for k, v in ragas_metrics.items()},
    })

comparison_df = pd.DataFrame(summary_rows)
comparison_df

C:\Users\Kao\AppData\Local\Temp\ipykernel_9768\2737781457.py:6: DeprecationWarning: Importing Faithfulness from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import Faithfulness
  from ragas.metrics import Faithfulness, LLMContextPrecisionWithoutReference
C:\Users\Kao\AppData\Local\Temp\ipykernel_9768\2737781457.py:6: DeprecationWarning: Importing LLMContextPrecisionWithoutReference from 'ragas.metrics' is deprecated and will be removed in v1.0. Please use 'ragas.metrics.collections' instead. Example: from ragas.metrics.collections import LLMContextPrecisionWithoutReference
  from ragas.metrics import Faithfulness, LLMContextPrecisionWithoutReference
C:\Users\Kao\AppData\Local\Temp\ipykernel_9768\2737781457.py:17: DeprecationWarning: LangchainLLMWrapper is deprecated and will be removed in a future version. Use llm_factory instead: from openai import OpenAI; from ragas.llms import llm_f

,prompt_version,sentiment_parsed/mean,extractive_lines_count/mean,llm_groundedness_and_format/mean,has_five_lines/mean,context_precision,faithfulness_full,faithfulness_extractive,n_extractive
0,v1,0.67,2.9,0.4448,0.57,0.5526,0.3678,0.5406,101
1,v2,1.00,3.0,0.4614,0.17,0.3894,0.1921,0.4494,101


In [5]:
# 6.) Inspect actual line-separated outputs from MLflow traces ($0 — no API calls)
import json
import re
import sys
from pathlib import Path

import mlflow
import pandas as pd

REPO_ROOT = Path.cwd().parent if (Path.cwd().parent / "app").is_dir() else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from evaluation.extractive import extractive_only, sentiment_label

mlflow.set_tracking_uri("http://localhost:5000")

# Inspect the exact trace shown in the MLflow UI screenshot
TRACE_ID = "tr-1c93b05c98ba05ac9444459de00ab567"
tr = mlflow.get_trace(TRACE_ID)
raw_resp = tr.data.response
if isinstance(raw_resp, str) and raw_resp.startswith('"'):
    raw_resp = json.loads(raw_resp)

print(f"Trace: {TRACE_ID}")
print("--- Raw repr (shows preserved \\n newlines) ---")
print(repr(raw_resp))
print("\n--- Rendered plain text ---")
print(raw_resp)

Trace: tr-1c93b05c98ba05ac9444459de00ab567
--- Raw repr (shows preserved \n newlines) ---
'Sentiment: positive; confidence 0.70; driving phrase: ราคานี้ได้ระดับนี้ก็เยี่ยมมากแล้วครับ\nIssues: - [portion] ไม่ค่อยได้สัมผัสของรังนกเท่าที่ควร\nPrecedent: portion — not in corpus; source id: not_in_corpus\nActions: protect: รักษาคุณภาพอาหารและความหลากหลายของเมนูรังนก; protect: คงราคาคุ้มค่า\nPriority: low; เน้นรักษาคุณภาพอาหารและการบริการให้คงที่'

--- Rendered plain text ---
Sentiment: positive; confidence 0.70; driving phrase: ราคานี้ได้ระดับนี้ก็เยี่ยมมากแล้วครับ
Issues: - [portion] ไม่ค่อยได้สัมผัสของรังนกเท่าที่ควร
Precedent: portion — not in corpus; source id: not_in_corpus
Actions: protect: รักษาคุณภาพอาหารและความหลากหลายของเมนูรังนก; protect: คงราคาคุ้มค่า
Priority: low; เน้นรักษาคุณภาพอาหารและการบริการให้คงที่


In [6]:
# Compare v1 vs v2 response format across all traces saved in MLflow
exp = mlflow.get_experiment_by_name("restaurant_rag/prompt_registry_eval")
runs = mlflow.search_runs(experiment_ids=[exp.experiment_id], order_by=["start_time DESC"])

latest_runs = {}
for v in [1, 2]:
    matched = runs[runs["params.prompt_version"] == str(v)]
    latest_runs[v] = matched.iloc[0]["run_id"]

_NUM_RE = re.compile(r"^\s*([1-5])[.)]\s+", re.M)
_SEC_RE = re.compile(
    r"^\s*(?:\d[.)]\s*)?(sentiment|issues|precedent|actions|priority)\s*:", re.I | re.M
)


def format_stats(text: str) -> dict:
    lines = [ln for ln in (text or "").splitlines() if ln.strip()]
    nums = set(_NUM_RE.findall(text or ""))
    secs = {m.lower() for m in _SEC_RE.findall(text or "")}
    ext_lines = [ln for ln in extractive_only(text or "").splitlines() if ln.strip()]
    return {
        "all_5_sections": secs == {"sentiment", "issues", "precedent", "actions", "priority"},
        "numbered_1_to_5": nums == {"1", "2", "3", "4", "5"},
        "exactly_5_lines": len(lines) == 5,
        "n_lines": len(lines),
        "extractive_3_lines": len(ext_lines) == 3,
        "sentiment_parsed": sentiment_label(text or "") != "unparsed",
    }


traces_by_version = {}
format_rows = []

for v, rid in latest_runs.items():
    df_t = mlflow.search_traces(locations=[exp.experiment_id], run_id=rid, max_results=200)
    # Keep one trace per unique review (drops the extra pre-eval validation trace)
    df_t = df_t.assign(
        review=df_t["request"].apply(lambda r: r.get("review", "") if isinstance(r, dict) else str(r)),
        output=df_t["response"].apply(lambda s: s if isinstance(s, str) else str(s)),
    ).drop_duplicates(subset=["review"], keep="last").reset_index(drop=True)
    traces_by_version[v] = df_t

    stats_df = pd.DataFrame([format_stats(t) for t in df_t["output"]])
    format_rows.append({
        "version": f"v{v}",
        "n_traces": len(df_t),
        "all_5_sections_rate": round(stats_df["all_5_sections"].mean(), 3),
        "numbered_1_to_5_rate": round(stats_df["numbered_1_to_5"].mean(), 3),
        "exactly_5_lines_rate": round(stats_df["exactly_5_lines"].mean(), 3),
        "avg_lines": round(stats_df["n_lines"].mean(), 2),
        "extractive_3_lines_rate": round(stats_df["extractive_3_lines"].mean(), 3),
        "sentiment_parsed_rate": round(stats_df["sentiment_parsed"].mean(), 3),
    })

pd.DataFrame(format_rows)

,version,n_traces,all_5_sections_rate,numbered_1_to_5_rate,exactly_5_lines_rate,avg_lines,extractive_3_lines_rate,sentiment_parsed_rate
0,v1,100,0.93,0.57,0.81,6.52,0.93,0.67
1,v2,100,1.00,0.17,0.89,5.64,1.00,1.00


In [7]:
# Print side-by-side v1 vs v2 outputs with actual line breaks for the first N reviews
SAMPLE_N = 5
merged = traces_by_version[1][["review", "output"]].merge(
    traces_by_version[2][["review", "output"]],
    on="review",
    suffixes=("_v1", "_v2"),
)

for i, row in merged.head(SAMPLE_N).iterrows():
    print("=" * 80)
    print(f"[{i + 1}] Review: {row['review'][:140].replace(chr(10), ' ')}...")
    print("\n--- Prompt v1 ---")
    print(row["output_v1"])
    print("\n--- Prompt v2 ---")
    print(row["output_v2"])
    print()

[1] Review: เมนูข้าวหน้าหมูย่างแบบเผ็ด เมนูนี้มีหมูย่างกับซอสรสเผ็ดโรยหน้ามาด้วยกระเจี๊ยบเหนียวๆ แหยะๆ ข้าวญี่ปุ่นเองก็หุงแฉะมาก ประกอบกับกระเจี๊ยบสับแห...

--- Prompt v1 ---
Sentiment: Mixed; ความมั่นใจ 0.60 — หลายเมนูอร่อย โดยเฉพาะเส้นหมี่ XO และข้าวผัดไต้หวัน แต่ ข้าวต้มหอยนางรม ไม่ค่อยอร่อย

Issues: - [taste] ข้าวต้มหอยนางรม ไม่ค่อยอร่อย

Precedent: - Recurring: ข้าวหุงแฉะ (จากข้าวหน้าหมูย่าง) — source: past_reviews; - Not in corpus: ข้าวต้มหอยนางรมไม่ค่อยอร่อย

Actions: - verify: ปรับปรุงการหุงข้าว/ควบคุม texture ของข้าวในเมนูที่มีข้าว; - verify: ตรวจสอบสูตรข้าวต้มและความสดของหอยนางรม, เพิ่มทางเลือกท็อปปิ้ง; - Protect: รักษความหลากหลายและคุณภาพอาหารที่แข็งแกร่งในเมนูที่ได้รับชม

Priority: Med; เพราะมีทั้งข้อเสียน้อยแต่เชิงคุณภาพอาหารที่เกี่ยวกับข้าวและหอยนางรม

--- Prompt v2 ---
1) Sentiment: negative; confidence 0.75; driving phrase: รสชาติแย่มากกกกก, ข้าวหุงแฉะมาก  
2) Issues: - [taste] รสชาติเมนูข้าวหน้าหมูย่างแย่ - [taste] ข้าวญี่ปุ่นหุงแฉะมาก - [taste] กระเจี๊ยบสับแหยะๆ ทำให้